# 06 — Optimisers, schedules, regularisation

## Roadmap
1. SGD — the simplest possible optimiser, math + code
2. SGD with **momentum**
3. **Adam** — adaptive learning rates per parameter
4. **Learning-rate schedules** — warmup, cosine, step decay
5. **Batch normalization** — what it does, why it helps
6. **Dropout** + **L2 regularisation**
7. When to use which


In [ ]:
# === Bootstrap (safe to re-run) ===
# Installs minimal deps and downloads tiny CIFAR-10 subset for any notebook
# that needs it. The from-scratch notebooks deliberately avoid importing the
# project package so they run anywhere (Colab, plain Python, Jupyter).
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "numpy", "matplotlib", "pillow", "torchvision"], check=True)
import numpy as np
import matplotlib.pyplot as plt
np.random.seed(0)
print("numpy", np.__version__)


## 1. Plain SGD

$$\theta_{t+1} = \theta_t - \eta \nabla \mathcal{L}(\theta_t)$$

Take a step in the direction of steepest descent, with step size $\eta$ (learning rate).

Problem: noisy gradient causes wild oscillation when the loss surface is a ravine.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Toy 2-D loss: a ravine
def loss(p): return 0.5 * (p[0]**2 + 10 * p[1]**2)
def grad(p): return np.array([p[0], 10 * p[1]])

def sgd_path(lr=0.1, n_steps=40):
    p = np.array([5.0, 1.5])
    hist = [p.copy()]
    for _ in range(n_steps):
        p = p - lr * grad(p)
        hist.append(p.copy())
    return np.array(hist)

path = sgd_path(lr=0.1)
fig, ax = plt.subplots(figsize=(7, 4))
X, Y = np.meshgrid(np.linspace(-6, 6, 100), np.linspace(-3, 3, 100))
Z = 0.5 * (X**2 + 10*Y**2)
ax.contour(X, Y, Z, levels=20, alpha=0.5)
ax.plot(path[:, 0], path[:, 1], "o-r", markersize=3)
ax.set_title("SGD bouncing across the ravine"); ax.grid(True)
plt.show()


## 2. Momentum

Instead of just taking the gradient as the step, accumulate a **velocity** that's an exponential moving average of past gradients:

$$v_{t+1} = \mu v_t + \nabla \mathcal{L}(\theta_t)$$
$$\theta_{t+1} = \theta_t - \eta v_{t+1}$$

`μ` (typically 0.9) damps oscillation: gradients that flip back and forth average to zero; gradients that point consistently in one direction reinforce.


In [ ]:
def sgd_momentum_path(lr=0.1, mu=0.9, n_steps=40):
    p = np.array([5.0, 1.5]); v = np.zeros_like(p)
    hist = [p.copy()]
    for _ in range(n_steps):
        v = mu * v + grad(p)
        p = p - lr * v
        hist.append(p.copy())
    return np.array(hist)

paths = {"SGD": sgd_path(lr=0.1), "SGD+momentum": sgd_momentum_path(lr=0.1)}
fig, ax = plt.subplots(figsize=(7, 4))
ax.contour(X, Y, Z, levels=20, alpha=0.5)
for name, p in paths.items():
    ax.plot(p[:, 0], p[:, 1], "o-", markersize=3, label=name)
ax.legend(); ax.set_title("Momentum smooths the path"); ax.grid(True); plt.show()


## 3. Adam — different learning rate per parameter

Adam keeps two moving averages: first moment (mean of gradients) and second moment (mean of *squared* gradients). It divides by `sqrt(second moment)` so parameters with consistently large gradients take **smaller** steps, and rare-but-important gradients get a chance to move.

$$m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t$$
$$v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2$$
$$\theta_{t+1} = \theta_t - \eta \frac{\hat{m_t}}{\sqrt{\hat{v_t}} + \epsilon}$$

(The hats are bias corrections you need only in the first few iterations.)


In [ ]:
def adam_path(lr=0.3, b1=0.9, b2=0.999, eps=1e-8, n_steps=40):
    p = np.array([5.0, 1.5])
    m = np.zeros_like(p); v = np.zeros_like(p)
    hist = [p.copy()]
    for t in range(1, n_steps + 1):
        g = grad(p)
        m = b1 * m + (1 - b1) * g
        v = b2 * v + (1 - b2) * g**2
        m_hat = m / (1 - b1**t)
        v_hat = v / (1 - b2**t)
        p = p - lr * m_hat / (np.sqrt(v_hat) + eps)
        hist.append(p.copy())
    return np.array(hist)

paths = {"SGD": sgd_path(lr=0.1),
          "SGD+momentum": sgd_momentum_path(lr=0.1),
          "Adam": adam_path(lr=0.3)}
fig, ax = plt.subplots(figsize=(7, 4))
ax.contour(X, Y, Z, levels=20, alpha=0.5)
for name, p in paths.items():
    ax.plot(p[:, 0], p[:, 1], "o-", markersize=3, label=name)
ax.legend(); ax.set_title("Adam converges with much fewer steps"); plt.show()


**Rule of thumb:**

| Task | First try |
|---|---|
| Brand new model, you have no idea | **Adam(3e-4)** — almost always trains *something* |
| Image classifier, plenty of data | **SGD + momentum** with cosine schedule — often gets best final accuracy |
| Transformer / language model | **AdamW** (Adam with decoupled weight decay) |

## 4. Learning-rate schedules

Constant LR is rarely optimal. Two common schedules:

- **Step decay** — multiply LR by 0.1 every 30 epochs.
- **Cosine annealing** — smoothly decrease from `lr_max` to `lr_min` over the full run.
- **Warmup** — start tiny, ramp up linearly for a few hundred steps. Critical for very large LR.


In [ ]:
def cosine_schedule(step, total_steps, lr_max=1.0, lr_min=0.0):
    return lr_min + 0.5 * (lr_max - lr_min) * (1 + np.cos(np.pi * step / total_steps))

steps = np.arange(0, 200)
lrs = [cosine_schedule(s, 200, lr_max=1.0) for s in steps]
plt.figure(figsize=(7, 3))
plt.plot(steps, lrs); plt.title("cosine learning-rate schedule"); plt.grid(True); plt.show()


## 5. Batch normalization

For each mini-batch, normalize the activations of a layer to mean 0, var 1 (per channel for conv). Then apply a learnable scale and shift:

$$\hat{x} = \frac{x - \mu_B}{\sqrt{\sigma_B^2 + \epsilon}}, \quad y = \gamma \hat{x} + \beta$$

Effects:

1. Stabilizes gradient magnitudes → can use a higher learning rate.
2. Adds mild regularisation noise (the batch statistics are noisy).
3. Lets you train much deeper networks.

Used everywhere in CNNs since 2015.

## 6. Dropout and L2

**Dropout** — during training, randomly zero out fraction $p$ of activations. Forces the network to not rely on any single neuron.

**L2 weight decay** — add $\lambda \sum w^2$ to the loss. Pulls weights toward zero, prevents huge co-adapted weights.

Both reduce overfitting. Standard amounts: dropout 0.1–0.5, L2 1e-4 for vision models.

**Next:** measure how well any of these are actually working — metrics deep-dive.
